In [ ]:
import numpy as np
import pandas as pd
import ruptures as rpt
import os
import random
from kernelcpd import binseg
from joblib import Parallel, delayed

In [ ]:
np.random.seed(123)
random.seed(123)

In [ ]:
n_signals = 500
signal_length = 200
n_per_type = int(n_signals/5)

signals = []
true_bkps = []
signal_types = []

for signal_type in ["mean", "variance", "frequency", "slope", "correlation"]:
    for _ in range(n_per_type):
        shift = np.random.uniform(0.03, 1.0)
        y = np.zeros(signal_length)
        t_star = np.random.randint(80, 121)

        if signal_type == "mean":
            sd = np.random.uniform(2, 4)
            y[:t_star] = np.random.normal(-1, sd, t_star)
            y[t_star:] = np.random.normal(1, sd, signal_length - t_star)

        elif signal_type == "variance":
            v = np.random.uniform(0.8, 1.2)
            y[:t_star] = np.random.normal(0, np.sqrt(v), t_star)
            y[t_star:] = np.random.normal(0, np.sqrt(v + shift), signal_length - t_star)

        elif signal_type == "frequency":
            t = np.arange(t_star)
            y[:t_star] = np.cos(2 * np.pi * 0.05 * t)
            t2 = np.arange(signal_length - t_star)
            y[t_star:] = np.cos(2 * np.pi * (0.05 + shift) * t2)
            y = y + np.random.normal(0, 0.1, signal_length)

        elif signal_type == "slope":
            slope_before = 0.2
            slope_after  = 0.2
            y[:t_star] = np.linspace(0, 1, t_star) * slope_before
            y[t_star:] = np.linspace(0, 1, signal_length - t_star) * (slope_before + shift)
            y = y + np.random.normal(0, 0.1, signal_length)

        elif signal_type == "correlation":
            f = np.random.uniform(0.3, 0.9)
            y[0]=0
            for t in range(1, signal_length):
                if t < t_star:
                    y[t] = f * y[t-1]
                else:
                    y[t] = -f * y[t-1]
            y = y + np.random.normal(0, 0.1, signal_length)

        signals.append(y)
        true_bkps.append(t_star)
        signal_types.append(signal_type)

In [ ]:
# # Save all signals
# for i, y in enumerate(signals):
#     np.save(f"signals/signal_{i + 1}.npy", y)

In [ ]:
models = ["linear", "cosine", "rbf", "proposed"]
signal_classes = ["mean", "variance", "frequency", "slope", "correlation"]

def detect_all_models(y):
    """Run all models once for a single signal."""
    out = {}
    out["linear"] = rpt.KernelCPD(kernel="linear").fit(y).predict(n_bkps=1)[0]
    out["cosine"] = rpt.KernelCPD(kernel="cosine").fit(y).predict(n_bkps=1)[0]
    out["rbf"] = rpt.KernelCPD(kernel="rbf").fit(y).predict(n_bkps=1)[0]
    out["proposed"] = (
            binseg(
                sequence=y.reshape(-1,1),
                n_changepoints=1,
                model="srbf",
                T=np.random.randint(2, 10),
                laplace_option=np.random.choice(["heat", "dirichlet", "tps"]),
                laplace_hyperparameter=np.random.randint(1, 2)
            )[0] + 1
        )

    return out

In [ ]:
# Run detection in parallel over signals
detections = Parallel(n_jobs=-1, verbose=5)(
    delayed(detect_all_models)(y) for y in signals
)

In [ ]:
acc_df = pd.DataFrame(
    0.00,
    index=models,
    columns=signal_classes
)

for model in models:
    for stype in signal_classes:
        correct = 0
        for i in range(len(signals)):
            if signal_types[i] != stype:
                continue

            cp = detections[i][model]
            if cp >= 0 and abs(cp - true_bkps[i]) <= 20:
                correct += 1

        acc_df.loc[model, stype] = correct / n_per_type

In [ ]:
acc_df.to_csv("results/acc_df.csv")